# Retrieval Playground — coding workshop (beginner)

About 20–25 minutes, individual. No GPU and no downloads needed: any Colab runtime works.

Sparse retrieval ranks documents by word overlap; dense retrieval ranks them by embedding
similarity. Here you build a tiny version of both by hand, over five toy documents. Notebook
3's Part 2 then runs the same comparison over the 26-document course corpus.

## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

## Part A — Sparse retrieval, by hand

Five toy documents on topics from this course. No library and no TF-IDF yet: count how
many query words appear in each document, and rank by that count. This is the crudest
possible sparse retrieval, which makes both the mechanism and its failure mode easy to see.

### ✏️ Exercise 1 — Word-overlap scoring

Complete `word_overlap_score`: the number of distinct words that the query and the
document share, ignoring case.

<details><summary><b>Hints</b> (click to open)</summary>

- `set(...)` keeps each word once.
- `a & b` is the set of elements in both `a` and `b`.

</details>

In [ ]:
TOY_DOCS = {
    "doc_carbon": "training a model produces carbon emissions measured once",
    "doc_energy": "running a model costs energy every single time it answers a query",
    "doc_water": "data centers consume water for cooling as well as electricity",
    "doc_retrieval": "a retriever finds relevant documents before the model reads them",
    "doc_quantize": "quantization stores numbers with fewer bits to save memory",
}

def word_overlap_score(query, doc_text):
    query_words = set(query.lower().split())
    doc_words = set(doc_text.lower().split())
    return len(___ & ___)


def rank_by_overlap(query):
    scores = {name: word_overlap_score(query, text) for name, text in TOY_DOCS.items()}
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        shared = set(query.lower().split()) & set(TOY_DOCS[name].split())
        print(f"{name:14s} {score} shared: {sorted(shared)}")


QUERY_A = "how much energy does running a model use"
rank_by_overlap(QUERY_A)

In [ ]:
# Check for Exercise 1. Run this cell as is.
assert word_overlap_score("Energy model", "a model uses energy") == 2
assert word_overlap_score("a a a", "a") == 1, "count each shared word once"
assert word_overlap_score("water", "energy") == 0
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
TOY_DOCS = {
    "doc_carbon": "training a model produces carbon emissions measured once",
    "doc_energy": "running a model costs energy every single time it answers a query",
    "doc_water": "data centers consume water for cooling as well as electricity",
    "doc_retrieval": "a retriever finds relevant documents before the model reads them",
    "doc_quantize": "quantization stores numbers with fewer bits to save memory",
}

def word_overlap_score(query, doc_text):
    query_words = set(query.lower().split())
    doc_words = set(doc_text.lower().split())
    return len(query_words & doc_words)


def rank_by_overlap(query):
    scores = {name: word_overlap_score(query, text) for name, text in TOY_DOCS.items()}
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        shared = set(query.lower().split()) & set(TOY_DOCS[name].split())
        print(f"{name:14s} {score} shared: {sorted(shared)}")


QUERY_A = "how much energy does running a model use"
rank_by_overlap(QUERY_A)
```

</details>

## Try it yourself: a paraphrased query

`QUERY_B` asks essentially the same question as `QUERY_A` in different words, with no
`"energy"` and no `"model"`. Run the cell and see what word-overlap scoring does with it.

In [ ]:
QUERY_B = "what does it cost in electricity to get an answer"
rank_by_overlap(QUERY_B)

**Reflection:** Compare the two rankings, for `QUERY_A` and `QUERY_B`. `QUERY_B` asks
essentially the same thing as `QUERY_A` in different words. Did word-overlap scoring still
find `doc_energy`? Look at the `shared` words: which of them tell you anything about the
topic? What would need to be true about a query and a document for this scoring method to
work?

## Part A, continued — weighting by rarity (TF-IDF)

Word-overlap scoring treats every match the same: one shared word is one point, whether the
word is `"a"` or `"energy"`. TF-IDF scores a word higher the rarer it is across the
collection, so a distinctive word like `energy` counts for more than a common one like
`model`.

Two ingredients:
- **tf** (term frequency): how many times does the word appear in *this* document?
- **idf** (inverse document frequency): how rare is the word across *all* the documents?
  `idf(word) = log(number of documents / documents containing the word)`

Score = tf × idf. The cell below computes both for two words in `doc_energy`: `model`
(in 3 of the 5 documents) and `energy` (in only 1).

### ✏️ Exercise 2 — TF-IDF by hand

Three parts:

1. `idf(word, docs)` = log(number of documents / number of documents that contain `word`).
2. `tf(word, doc_text)` = how many times `word` occurs in the document.
3. At the end of the cell, `tfidf_score(query, doc_text)` adds up tf × idf over the
   distinct query words. It skips words that occur in no document, since their idf would
   divide by zero. The cell then ranks the five documents for `QUERY_A` and `QUERY_B`.

<details><summary><b>Hints</b> (click to open)</summary>

- `math.log` is the natural logarithm.
- `list.count(x)` counts how many times `x` occurs in a list.
- For each query word: `tf(word, doc_text) * idf(word, docs)`.

</details>

In [ ]:
import math

def idf(word, docs):
    doc_count = sum(1 for text in docs.values() if ___ in text.lower().split())
    return math.log(___ / ___)

def tf(word, doc_text):
    return doc_text.lower().split().count(___)

for word in ["model", "energy"]:
    print(f"idf('{word}') = {idf(word, TOY_DOCS):.3f}")

print("\nInside doc_energy, both words appear exactly once (tf = 1 for both):")
for word in ["model", "energy"]:
    score = tf(word, TOY_DOCS["doc_energy"]) * idf(word, TOY_DOCS)
    print(f"  tf-idf('{word}', doc_energy) = {score:.3f}")


def tfidf_score(query, doc_text, docs=TOY_DOCS):
    total = 0.0
    for word in set(query.lower().split()):
        if any(word in text.lower().split() for text in docs.values()):
            total += ___ * ___
    return total


for query in [QUERY_A, QUERY_B]:
    print(f"\n{query}")
    for name, text in sorted(TOY_DOCS.items(), key=lambda kv: -tfidf_score(query, kv[1])):
        print(f"  {name:14s} {tfidf_score(query, text):.3f}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert abs(idf("model", TOY_DOCS) - math.log(5 / 3)) < 1e-9, "'model' occurs in 3 of the 5 documents"
assert abs(idf("energy", TOY_DOCS) - math.log(5)) < 1e-9
assert tf("a", "a b a") == 2
assert max(TOY_DOCS, key=lambda n: tfidf_score(QUERY_A, TOY_DOCS[n])) == "doc_energy"
assert tfidf_score("zzz", TOY_DOCS["doc_energy"]) == 0
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
import math

def idf(word, docs):
    doc_count = sum(1 for text in docs.values() if word in text.lower().split())
    return math.log(len(docs) / doc_count)

def tf(word, doc_text):
    return doc_text.lower().split().count(word)

for word in ["model", "energy"]:
    print(f"idf('{word}') = {idf(word, TOY_DOCS):.3f}")

print("\nInside doc_energy, both words appear exactly once (tf = 1 for both):")
for word in ["model", "energy"]:
    score = tf(word, TOY_DOCS["doc_energy"]) * idf(word, TOY_DOCS)
    print(f"  tf-idf('{word}', doc_energy) = {score:.3f}")


def tfidf_score(query, doc_text, docs=TOY_DOCS):
    total = 0.0
    for word in set(query.lower().split()):
        if any(word in text.lower().split() for text in docs.values()):
            total += tf(word, doc_text) * idf(word, docs)
    return total


for query in [QUERY_A, QUERY_B]:
    print(f"\n{query}")
    for name, text in sorted(TOY_DOCS.items(), key=lambda kv: -tfidf_score(query, kv[1])):
        print(f"  {name:14s} {tfidf_score(query, text):.3f}")
```

</details>

**Reflection:** Why does `energy` score higher than `model`, even though both appear
exactly once in `doc_energy`? Now run `idf("it", TOY_DOCS)` or `idf("to", TOY_DOCS)`
yourself. What do you get, and why might that number be misleading on a collection this
small?

## Part B — Dense retrieval, by hand

Dense retrieval compares vectors instead of strings. A real system gets its vectors from a
trained embedding model; here we use hand-picked 2-D toy vectors, the same approach as the
lecture's aunt/uncle/car example, so every number is visible. A document's vector is the
average of the vectors of the words we defined.

### ✏️ Exercise 3 — Cosine similarity and text vectors

Complete `cosine_sim` and `text_vector`. A text's vector is the average of the toy
vectors of its words that have one.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.linalg.norm(v)` is the length of `v`.
- Averaging a list of 2-D vectors element by element: `np.mean(list_of_vectors, axis=0)`.

</details>

In [ ]:
import numpy as np

TOY_WORD_VECTORS = {
    "energy":      np.array([4, 2]),
    "electricity": np.array([4, 3]),
    "carbon":      np.array([3, 4]),
    "water":       np.array([1, 4]),
    "retriever":   np.array([0, 5]),
    "memory":      np.array([2, -3]),
}

def cosine_sim(v1, v2):
    return float(np.dot(v1, v2) / (___ * ___))

def text_vector(text):
    words = [w for w in text.lower().split() if w in TOY_WORD_VECTORS]
    return np.mean([___ for w in words], axis=___)

TOY_DOC_VECTORS = {name: text_vector(text) for name, text in TOY_DOCS.items()}
for name, vec in TOY_DOC_VECTORS.items():
    print(f"{name:14s} {vec}")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert abs(cosine_sim(np.array([4, 2]), np.array([3, 3])) - 0.949) < 1e-3, "Session 1 example: cos(aunt, uncle)"
assert np.allclose(text_vector("energy and water"), [2.5, 3.0]), text_vector("energy and water")
assert np.allclose(TOY_DOC_VECTORS["doc_energy"], [4, 2])
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import numpy as np

TOY_WORD_VECTORS = {
    "energy":      np.array([4, 2]),
    "electricity": np.array([4, 3]),
    "carbon":      np.array([3, 4]),
    "water":       np.array([1, 4]),
    "retriever":   np.array([0, 5]),
    "memory":      np.array([2, -3]),
}

def cosine_sim(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def text_vector(text):
    words = [w for w in text.lower().split() if w in TOY_WORD_VECTORS]
    return np.mean([TOY_WORD_VECTORS[w] for w in words], axis=0)

TOY_DOC_VECTORS = {name: text_vector(text) for name, text in TOY_DOCS.items()}
for name, vec in TOY_DOC_VECTORS.items():
    print(f"{name:14s} {vec}")
```

</details>

## Rank the paraphrased query by cosine similarity

Same `QUERY_B` as in Part A, now embedded with the same averaging. The only word it has a
toy vector for is `"electricity"`, so that is its vector.

### ✏️ Exercise 4 — Rank by cosine similarity

Score every document against the query vector, in a dict from document name to
cosine similarity.

<details><summary><b>Hints</b> (click to open)</summary>

- Loop over `TOY_DOC_VECTORS.items()`: each item is a name and its vector.

</details>

In [ ]:
query_b_vector = text_vector(QUERY_B)
print("QUERY_B vector:", query_b_vector, "\n")

dense_scores_b = {name: cosine_sim(___, ___) for name, vec in TOY_DOC_VECTORS.items()}
for name, score in sorted(dense_scores_b.items(), key=lambda x: -x[1]):
    print(f"{name:14s} cosine similarity {score:.3f}")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert set(dense_scores_b) == set(TOY_DOCS)
assert max(dense_scores_b, key=dense_scores_b.get) == "doc_energy"
assert abs(dense_scores_b["doc_carbon"] - 0.96) < 1e-9
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
query_b_vector = text_vector(QUERY_B)
print("QUERY_B vector:", query_b_vector, "\n")

dense_scores_b = {name: cosine_sim(query_b_vector, vec) for name, vec in TOY_DOC_VECTORS.items()}
for name, score in sorted(dense_scores_b.items(), key=lambda x: -x[1]):
    print(f"{name:14s} cosine similarity {score:.3f}")
```

</details>

**Reflection:** Compare this ranking to `QUERY_B`'s word-overlap ranking in Part A. Did
`doc_energy` do better? What is different about how this score was computed? The toy
vectors were hand-picked so that "electricity" sits close to "energy". Where does that
closeness come from in a trained embedding model? (Think back to the skip-gram lecture.)
Look at how close the top three scores are: what might that mean for a retriever? And
what does `doc_quantize`'s negative score tell you about the angle between the two
vectors?

## Code literacy check

Before you move on, pick at least two code cells above that you ran and read today, and
add `#` comments explaining in your own words what each meaningful line or block does and
*why* it's there, so that a classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini, whichever you have
   access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, check the code again: who's
   right?

This is a self-check on your own understanding: a way to find where your mental model of
the code breaks down against a second reader.